# CUDA 编译运行模板 (Colab / T4)

**用法**:
1. 菜单 `代码执行程序` → `更改运行时类型` → 硬件加速器选 **T4 GPU**。
2. 从上到下依次运行各 cell。
3. 在 `%%writefile` cell 里粘贴你的 kernel 代码,或用左侧文件面板上传 `.cu` + `common.h`。

> 本机(Mac Mini)不能跑 nvcc,所有编译运行都在这里做。编译目标 `-arch=sm_75`(T4, Turing)。

## 1. 确认 GPU 环境

In [ ]:
!nvidia-smi
!nvcc --version

## 2. 准备 common.h

两种方式二选一:
- **A. 上传**:用左侧文件面板把仓库里的 `week1/common.h` 拖上来。
- **B. 直接写**:运行下面这个 cell 生成一份(内容需与仓库保持同步)。

In [ ]:
%%writefile common.h
// 如果你选择方式 A(上传),就不要运行这个 cell,以免覆盖上传的版本。
// 方式 B:把仓库 week1/common.h 的完整内容粘到这里。
#ifndef COMMON_H
#define COMMON_H
// ... 粘贴 common.h 内容 ...
#endif

## 3. 写 kernel 代码

把你的程序写进 `main.cu`。记得每个 kernel 程序要有:**错误检查宏 + 边界检查 + CPU 端验证**。

In [ ]:
%%writefile main.cu
#include "common.h"
#include <cstdio>

// TODO: 在这里写你的 kernel 和 main()

int main() {
    printf("hello from host\n");
    return 0;
}

## 4. 编译

`-arch=sm_75` 对应 T4 (Turing)。如果要在 GTX 1050 Ti (sm_61) 上跑,注意别用 Turing+ 独有特性。
`-lineinfo` 让 profiler 能把指令映射回源码行;`-O2` 常规优化。

In [ ]:
!nvcc -arch=sm_75 -O2 -lineinfo main.cu -o main 2>&1

## 5. 运行

In [ ]:
!./main

## 6.(可选)compute-sanitizer 查内存越界 / 竞态

类比嵌入式里的 Valgrind / 硬件 MPU 报错。kernel 越界、未初始化读、竞态在这里能抓到。

In [ ]:
!compute-sanitizer --tool memcheck ./main